In [7]:
import pandas as pd
import numpy as np
import os
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
# import xgboost as xgb
# import lightgbm as lgb
from sklearn.neighbors import KNeighborsClassifier
# from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, recall_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

In [8]:
# 统计每列中每个值的数量
def stat_label_count(
    df,
    freq,
    dataset_training_path,
    columns=[
        "top_or_bottom",
        "top_or_bottom_stat",
        "top_bottom_volatility_stat",
        "top_or_bottom_stat_optimized",
        "top_or_bottom_optimized",
        "top_bottom_volatility_optimized",
    ],
    version="0.1",
):
    result = {}
    for column in columns:
        result[column] = df[column].value_counts()

    # 将统计结果转换为 DataFrame
    result_df = pd.DataFrame(result).fillna(0).astype(int)

    # 保存统计结果为 CSV 文件
    result_df.to_csv(
        f"{dataset_training_path}label_value_counts_{freq}_{version}.csv", index_label="value"
    )

    print("统计结果已保存为 value_counts.csv")

In [26]:
# 文件路径
root_path_DEV = "./stock_filestore/DEV/"
root_path_PROD = "./stock_filestore/PROD/"
folder_ds_entry = "dataset_entry/"
# folder_ds_tech_funda = "dataset_tech_funda/"
dataset_training_path = root_path_DEV + "dataset_training/"
dataset_training_stat_path = root_path_DEV + "dataset_training/stat/"

# dataset_entry_path = root_path_DEV + folder_ds_entry

In [4]:
freq = "D"
version = 0.2
df_d = pd.read_csv(f"{dataset_training_path}mapped_entry_{freq}.csv",)

In [13]:
freq = "M"
version = 0.2
df_m = pd.read_csv(f"{dataset_training_path}mapped_entry_{freq}.csv",)

In [15]:
df_m.describe()

,open,high,low,close_df1,pre_close,change,pct_chg,vol,amount,atr,...,total_mv,circ_mv,float_share_ratio,free_share_ratio,top_or_bottom,top_or_bottom_stat,top_bottom_volatility_stat,top_or_bottom_stat_optimized,top_or_bottom_optimized,top_bottom_volatility_optimized
count,778853.000000,778853.000000,778853.000000,778853.000000,778853.00000,778853.000000,778853.000000,7.788530e+05,7.788530e+05,703187.000000,...,7.788460e+05,7.788460e+05,778845.000000,777670.000000,778853.000000,778853.000000,778853.000000,778853.000000,778853.000000,778853.000000
mean,12.842080,14.413994,11.473223,12.745099,12.77477,-0.029671,1.478154,2.430937e+08,2.944557e+09,2.789383,...,1.484688e+06,1.013507e+06,0.672149,0.444233,0.665445,0.147127,0.091916,0.131013,0.533009,0.083152
std,24.015946,26.466184,21.654030,23.844332,23.86031,4.249480,24.066825,6.104519e+08,7.250877e+09,4.458520,...,7.218987e+06,4.915489e+06,0.294672,0.186502,0.817249,0.468781,0.376696,0.444901,0.776445,0.359348
min,0.090000,0.090000,0.090000,0.090000,0.09000,-484.600000,-95.730000,1.000000e+01,2.440000e+02,0.080000,...,1.196638e+03,1.196638e+03,0.009949,0.000406,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.830000,5.360000,4.400000,4.820000,4.81000,-0.600000,-7.420000,3.433458e+07,4.144539e+08,1.050000,...,2.449415e+05,1.228945e+05,0.398471,0.299194,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,8.010000,8.930000,7.260000,7.990000,7.99000,-0.020000,-0.410000,9.289316e+07,1.106200e+09,1.750000,...,4.431564e+05,2.899522e+05,0.711632,0.434418,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,14.000000,15.680000,12.570000,13.930000,13.96000,0.540000,7.450000,2.387698e+08,2.819105e+09,3.030000,...,9.393927e+05,6.697229e+05,0.993352,0.581809,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,2222.000000,2603.270000,2047.940000,2197.230000,2218.00000,300.120000,3484.850000,4.587650e+10,6.611574e+11,240.520000,...,5.768821e+08,2.786247e+08,1.003501,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000


In [14]:
stat_label_count(df_m, freq, dataset_training_stat_path)

统计结果已保存为 value_counts.csv


In [16]:
df_m.columns

Index(['ts_code_df1', 'trade_date', 'open', 'high', 'low', 'close_df1',
       'pre_close', 'change', 'pct_chg', 'vol', 'amount', 'atr', 'sl_atr',
       'sl_2atr', 'sl_3atr', 'sl_4atr', 'sl_5atr', 'pct_vol_chg',
       'pct_amount_chg', 'close_atr_diff', 'close_2atr_diff',
       'close_3atr_diff', 'close_4atr_diff', 'close_5atr_diff', 'pct_o2c',
       'lower_shadow', 'upper_shadow', 'dif', 'dea', 'bar', 'rsi_6', 'rsi_12',
       'rsi_24', 'k', 'd', 'j', 'mab_10', 'mab_25', 'mab_60', 'mab_120',
       'mab_200', 'turnover_rate', 'turnover_rate_f', 'volume_ratio', 'pe',
       'pe_ttm', 'pb', 'ps', 'ps_ttm', 'dv_ratio', 'dv_ttm', 'total_share',
       'float_share', 'free_share', 'total_mv', 'circ_mv', 'float_share_ratio',
       'free_share_ratio', 'top_or_bottom', 'top_or_bottom_stat',
       'top_bottom_volatility_stat', 'top_or_bottom_stat_optimized',
       'top_or_bottom_optimized', 'top_bottom_volatility_optimized'],
      dtype='object')

In [18]:
# 统计每个 ts_code 的空值数量
missing_counts = df_m['turnover_rate'].isna().groupby(df_m['ts_code_df1']).sum()

# 打印结果
print("每个 ts_code 的 turnover_rate 空值数量：")
print(missing_counts)
# 保存统计结果为 CSV 文件
missing_counts.to_csv(
    f"{dataset_training_path}to_nan_value_counts_{freq}.csv", index_label="value"
)

每个 ts_code 的 turnover_rate 空值数量：
ts_code_df1
000001.SZ    0
000002.SZ    0
000004.SZ    0
000005.SZ    0
000006.SZ    0
            ..
873703.BJ    0
873706.BJ    0
873726.BJ    0
873806.BJ    0
873833.BJ    0
Name: turnover_rate, Length: 5406, dtype: int64


In [19]:
def check_nan_impact_on_labels(df, columns_to_check, label_columns):
    # 初始化一个字典来存储结果
    results = []

    for column in columns_to_check:
        # 筛选出指定列为空值的行
        nan_rows = df[df[column].isnull()]

        for label_column in label_columns:
            # 统计标签列中每个值的数量
            value_counts = nan_rows[label_column].value_counts()
            for value, count in value_counts.items():
                results.append({
                    'column_with_nan': column,
                    'label_column': label_column,
                    'label_value': value,
                    'count': count
                })

    return pd.DataFrame(results)

In [23]:
def drop_nan_values(df, columns_to_drop=["close", "float_mv", "dv_ttm"]):
    # 删除包含 NaN 值的行
    df = df.drop(columns=columns_to_drop)
    df.dropna(
        how="any",
        axis=0,
        inplace=False,
    )
    return df


def drop_columns(df, columns_to_drop=["close", "float_mv", "dv_ttm"]):
    return df.drop(columns=columns_to_drop)


def map_labels(
    df,
    columns=[
        "top_or_bottom",
        "top_or_bottom_stat",
        "top_bottom_volatility_stat",
        "top_or_bottom_stat_optimized",
        "top_or_bottom_optimized",
        "top_bottom_volatility_optimized",
    ],
    class_mapping={"N": 0, "B": 1, "T": 2},
):
    # class_mapping = {"N":0, "B":1, "T":2}
    for column in columns:
        df[column] = df[column].map(class_mapping)
    return df

def check_nan_values(df):
    # 统计每列的空值数量
    nan_count = df.isnull().sum()

    # 统计每列的总数
    total_count = df.shape[0]

    # 计算每列空值占总数的比例
    nan_ratio = nan_count / total_count

    # 创建一个 DataFrame 来存储结果
    nan_stats = pd.DataFrame(
        {
            "column_name": nan_count.index,
            "nan_count": nan_count.values,
            "total_count": total_count,
            "nan_ratio": nan_ratio.values,
        }
    )

    print(nan_stats)
    return nan_stats

In [21]:
# 要检查的列
columns_to_check = [
    "change",
    "pct_chg",
    "vol",
    "atr",
    "pct_vol_chg",
    "pct_o2c",
    "lower_shadow",
    "upper_shadow",
    "dif",
    "dea",
    "bar",
    "rsi_6",
    "rsi_12",
    "rsi_24",
    "k",
    "d",
    "j",
    "turnover_rate",
    "turnover_rate_f",
    "volume_ratio",
    "pe",
    "pe_ttm",  # 当为负值时得到的数据为NaN
    "pb",
    "ps",
    "ps_ttm",
    "dv_ratio",
    # "dv_ttm",
    "total_share",
    "float_share",
    "free_share",
    "total_mv",
    "circ_mv",
    # "float_share_ratio",
    "free_share_ratio",
    "mab_10",
    "mab_25",
    "mab_60",
    "mab_120",
    "mab_200",
]

# 标签列
label_columns = [
    "top_or_bottom",
    "top_or_bottom_stat",
    "top_bottom_volatility_stat",
    "top_or_bottom_stat_optimized",
    "top_or_bottom_optimized",
    "top_bottom_volatility_optimized",
]

In [27]:
# 统计空值对标签列的影响
freq = "M"
version = "0.1"
nan_val = check_nan_values(df_m)
nan_val.to_csv(f"{dataset_training_stat_path}nan_val_entry_{freq}_{version}.csv", index=False)
print(nan_val)

nan_impact = check_nan_impact_on_labels(df_m, columns_to_check, label_columns)
# 保存结果到 CSV 文件
nan_impact.to_csv(f"{dataset_training_stat_path}nan_impact_{freq}_{version}.csv", index=False)
print(nan_impact)

                        column_name  nan_count  total_count  nan_ratio
0                       ts_code_df1          0       778853        0.0
1                        trade_date          0       778853        0.0
2                              open          0       778853        0.0
3                              high          0       778853        0.0
4                               low          0       778853        0.0
..                              ...        ...          ...        ...
59               top_or_bottom_stat          0       778853        0.0
60       top_bottom_volatility_stat          0       778853        0.0
61     top_or_bottom_stat_optimized          0       778853        0.0
62          top_or_bottom_optimized          0       778853        0.0
63  top_bottom_volatility_optimized          0       778853        0.0

[64 rows x 4 columns]
                        column_name  nan_count  total_count  nan_ratio
0                       ts_code_df1          0       7

In [28]:
# 将 trade_date 转换为日期类型
df_m['trade_date'] = pd.to_datetime(df_m['trade_date'])

# 按 ts_code 分组，并对每组数据按照 trade_date 排序后，从第 34 行开始提取数据
result = df_m.groupby('ts_code_df1', group_keys=False).apply(lambda group: group.sort_values('trade_date').iloc[34:])

# 打印结果
result.describe()

/var/folders/s6/srfc45q51jvbqnmg1xgbg9wr0000gn/T/ipykernel_4014/3560518986.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = df_m.groupby('ts_code_df1', group_keys=False).apply(lambda group: group.sort_values('trade_date').iloc[34:])


,trade_date,open,high,low,close_df1,pre_close,change,pct_chg,vol,amount,...,total_mv,circ_mv,float_share_ratio,free_share_ratio,top_or_bottom,top_or_bottom_stat,top_bottom_volatility_stat,top_or_bottom_stat_optimized,top_or_bottom_optimized,top_bottom_volatility_optimized
count,600947,600947.000000,600947.000000,600947.000000,600947.000000,600947.000000,600947.000000,600947.000000,6.009470e+05,6.009470e+05,...,6.009410e+05,6.009410e+05,600941.000000,600460.000000,600947.000000,600947.000000,600947.000000,600947.000000,600947.000000,600947.000000
mean,2016-03-06 22:20:17.889764608,11.587964,12.936034,10.423357,11.533493,11.566369,-0.032876,0.848689,2.902954e+08,3.305510e+09,...,1.607790e+06,1.239437e+06,0.769197,0.487091,0.667132,0.149168,0.088530,0.132090,0.533042,0.079538
min,1993-10-29 00:00:00,0.170000,0.190000,0.090000,0.110000,0.180000,-484.600000,-95.730000,5.000000e+02,2.790000e+03,...,2.543354e+03,2.073037e+03,0.012737,0.000406,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2011-06-30 00:00:00,4.820000,5.350000,4.410000,4.820000,4.820000,-0.550000,-7.110000,5.102866e+07,4.819550e+08,...,2.640263e+05,1.863130e+05,0.586249,0.359997,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2017-11-30 00:00:00,7.730000,8.590000,7.010000,7.710000,7.720000,-0.020000,-0.300000,1.265212e+08,1.259134e+09,...,4.851298e+05,3.777975e+05,0.854761,0.486146,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2022-01-28 00:00:00,12.800000,14.280000,11.510000,12.750000,12.790000,0.520000,7.360000,2.967753e+08,3.177354e+09,...,1.047298e+06,8.311023e+05,0.999873,0.617627,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,2025-02-28 00:00:00,2222.000000,2603.270000,2047.940000,2197.230000,2218.000000,300.120000,1342.420000,4.587650e+10,6.611574e+11,...,2.786247e+08,2.786247e+08,1.003501,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
std,NaN,23.581695,25.644217,21.612016,23.506961,23.477149,3.398315,15.293865,6.695821e+08,7.958673e+09,...,7.289183e+06,5.561247e+06,0.253201,0.180118,0.817897,0.473737,0.371679,0.448304,0.776353,0.353214


In [29]:
result["dv_ratio"] = result["dv_ratio"].fillna(0)
result["dv_ttm"] = result["dv_ttm"].fillna(0)
result.drop(columns=["mab_60", "mab_120", "mab_200"], inplace=True)
stat_label_count(result, freq, dataset_training_stat_path, version=0.2)

统计结果已保存为 value_counts.csv


In [31]:
result.describe()


,trade_date,open,high,low,close_df1,pre_close,change,pct_chg,vol,amount,...,total_mv,circ_mv,float_share_ratio,free_share_ratio,top_or_bottom,top_or_bottom_stat,top_bottom_volatility_stat,top_or_bottom_stat_optimized,top_or_bottom_optimized,top_bottom_volatility_optimized
count,600947,600947.000000,600947.000000,600947.000000,600947.000000,600947.000000,600947.000000,600947.000000,6.009470e+05,6.009470e+05,...,6.009410e+05,6.009410e+05,600941.000000,600460.000000,600947.000000,600947.000000,600947.000000,600947.000000,600947.000000,600947.000000
mean,2016-03-06 22:20:17.889764608,11.587964,12.936034,10.423357,11.533493,11.566369,-0.032876,0.848689,2.902954e+08,3.305510e+09,...,1.607790e+06,1.239437e+06,0.769197,0.487091,0.667132,0.149168,0.088530,0.132090,0.533042,0.079538
min,1993-10-29 00:00:00,0.170000,0.190000,0.090000,0.110000,0.180000,-484.600000,-95.730000,5.000000e+02,2.790000e+03,...,2.543354e+03,2.073037e+03,0.012737,0.000406,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2011-06-30 00:00:00,4.820000,5.350000,4.410000,4.820000,4.820000,-0.550000,-7.110000,5.102866e+07,4.819550e+08,...,2.640263e+05,1.863130e+05,0.586249,0.359997,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2017-11-30 00:00:00,7.730000,8.590000,7.010000,7.710000,7.720000,-0.020000,-0.300000,1.265212e+08,1.259134e+09,...,4.851298e+05,3.777975e+05,0.854761,0.486146,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2022-01-28 00:00:00,12.800000,14.280000,11.510000,12.750000,12.790000,0.520000,7.360000,2.967753e+08,3.177354e+09,...,1.047298e+06,8.311023e+05,0.999873,0.617627,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,2025-02-28 00:00:00,2222.000000,2603.270000,2047.940000,2197.230000,2218.000000,300.120000,1342.420000,4.587650e+10,6.611574e+11,...,2.786247e+08,2.786247e+08,1.003501,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
std,NaN,23.581695,25.644217,21.612016,23.506961,23.477149,3.398315,15.293865,6.695821e+08,7.958673e+09,...,7.289183e+06,5.561247e+06,0.253201,0.180118,0.817897,0.473737,0.371679,0.448304,0.776353,0.353214


In [30]:
stat_label_count(df_m, freq, dataset_training_stat_path, version=0.1)

统计结果已保存为 value_counts.csv


In [32]:
# 删除 'turnover_rate' 和 'volume_ratio' 列中包含空值的行
nan_columns = [
    "pb",
    "ps",
    "ps_ttm",
    "total_share",
    "float_share",
    "free_share",
    "free_share_ratio",
]
result = result.dropna(subset=nan_columns)
result.describe()

,trade_date,open,high,low,close_df1,pre_close,change,pct_chg,vol,amount,...,total_mv,circ_mv,float_share_ratio,free_share_ratio,top_or_bottom,top_or_bottom_stat,top_bottom_volatility_stat,top_or_bottom_stat_optimized,top_or_bottom_optimized,top_bottom_volatility_optimized
count,593873,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000,5.938730e+05,5.938730e+05,...,5.938730e+05,5.938730e+05,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000
mean,2016-03-22 11:16:22.785882112,11.658481,13.013678,10.487009,11.603083,11.636740,-0.033658,0.830377,2.917695e+08,3.335693e+09,...,1.623480e+06,1.251753e+06,0.770151,0.486489,0.667484,0.149436,0.088564,0.132293,0.533274,0.079549
min,1993-10-29 00:00:00,0.200000,0.250000,0.090000,0.110000,0.220000,-484.600000,-95.730000,1.030000e+04,5.005100e+04,...,4.865984e+03,2.502000e+03,0.012737,0.000406,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2011-07-29 00:00:00,4.860000,5.390000,4.450000,4.860000,4.860000,-0.550000,-7.100000,5.153589e+07,4.914579e+08,...,2.678496e+05,1.898556e+05,0.588721,0.359835,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2017-11-30 00:00:00,7.770000,8.640000,7.050000,7.760000,7.770000,-0.020000,-0.310000,1.273840e+08,1.277138e+09,...,4.909547e+05,3.824202e+05,0.855980,0.485667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2022-01-28 00:00:00,12.870000,14.360000,11.580000,12.810000,12.850000,0.520000,7.320000,2.984055e+08,3.209641e+09,...,1.058700e+06,8.404928e+05,0.999874,0.616797,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,2025-02-28 00:00:00,2222.000000,2603.270000,2047.940000,2197.230000,2218.000000,300.120000,1032.670000,4.587650e+10,6.611574e+11,...,2.786247e+08,2.786247e+08,1.003501,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
std,NaN,23.706524,25.779307,21.727023,23.631586,23.601359,3.415296,15.114554,6.724361e+08,7.999625e+09,...,7.330688e+06,5.592922e+06,0.252646,0.179690,0.817997,0.474386,0.371985,0.448862,0.776481,0.353448


系统自动标记的日期截止为2025-3-5号

In [33]:
version = 0.1
freq = 'M'
# df_test_d = pd.read_csv(f"{dataset_training_path}test_dataset_{freq}_{version}.csv",)

# 将 trade_date 列转换为日期类型
result['trade_date'] = pd.to_datetime(result['trade_date'])

# 删除 trade_date 大于 2025-03-05 的记录
result = result[result['trade_date'] <= pd.to_datetime('2025-03-05')]
result.describe()

,trade_date,open,high,low,close_df1,pre_close,change,pct_chg,vol,amount,...,total_mv,circ_mv,float_share_ratio,free_share_ratio,top_or_bottom,top_or_bottom_stat,top_bottom_volatility_stat,top_or_bottom_stat_optimized,top_or_bottom_optimized,top_bottom_volatility_optimized
count,593873,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000,5.938730e+05,5.938730e+05,...,5.938730e+05,5.938730e+05,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000,593873.000000
mean,2016-03-22 11:16:22.785882112,11.658481,13.013678,10.487009,11.603083,11.636740,-0.033658,0.830377,2.917695e+08,3.335693e+09,...,1.623480e+06,1.251753e+06,0.770151,0.486489,0.667484,0.149436,0.088564,0.132293,0.533274,0.079549
min,1993-10-29 00:00:00,0.200000,0.250000,0.090000,0.110000,0.220000,-484.600000,-95.730000,1.030000e+04,5.005100e+04,...,4.865984e+03,2.502000e+03,0.012737,0.000406,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2011-07-29 00:00:00,4.860000,5.390000,4.450000,4.860000,4.860000,-0.550000,-7.100000,5.153589e+07,4.914579e+08,...,2.678496e+05,1.898556e+05,0.588721,0.359835,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2017-11-30 00:00:00,7.770000,8.640000,7.050000,7.760000,7.770000,-0.020000,-0.310000,1.273840e+08,1.277138e+09,...,4.909547e+05,3.824202e+05,0.855980,0.485667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2022-01-28 00:00:00,12.870000,14.360000,11.580000,12.810000,12.850000,0.520000,7.320000,2.984055e+08,3.209641e+09,...,1.058700e+06,8.404928e+05,0.999874,0.616797,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,2025-02-28 00:00:00,2222.000000,2603.270000,2047.940000,2197.230000,2218.000000,300.120000,1032.670000,4.587650e+10,6.611574e+11,...,2.786247e+08,2.786247e+08,1.003501,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
std,NaN,23.706524,25.779307,21.727023,23.631586,23.601359,3.415296,15.114554,6.724361e+08,7.999625e+09,...,7.330688e+06,5.592922e+06,0.252646,0.179690,0.817997,0.474386,0.371985,0.448862,0.776481,0.353448


In [ ]:
# 定义分割函数
def split_train_test(group, train_ratio=0.7):
    # 按 trade_date 排序
    group = group.sort_values("trade_date")
    # 计算分割点
    split_index = int(len(group) * train_ratio)
    # 分割成训练集和测试集
    train = group.iloc[:split_index]
    test = group.iloc[split_index:]
    return train, test


def get_train_test_data(df, group_by="ts_code", train_ratio=0.8, freq="D", version="0.1"):
    # 按 code 分组
    groups = df.groupby(group_by)
    # 对每个分组应用分割函数
    res = [split_train_test(group, train_ratio) for name, group in groups]
    # 拆分训练集和测试集
    train = pd.concat([t[0] for t in res])
    test = pd.concat([t[1] for t in res])

    train.to_csv(f"{dataset_training_path}train_dataset_{freq}_{version}.csv", index=False)
    test.to_csv(f"{dataset_training_path}test_dataset_{freq}_{version}.csv", index=False)
    print(f"train_dataset_{freq}_{version}.csv and test_dataset_{freq}_{version}.csv saved")
    return train, test

In [36]:
# 保存数据集
freq = "M"
version = 0.1
train, test = get_train_test_data(result, group_by="ts_code_df1", train_ratio=0.7, freq=freq, version=version)
# train.to_csv(f"{training_path}train_dataset_{freq}_{version}.csv", index=False)
# test.to_csv(f"{training_path}test_dataset_{freq}_{version}.csv", index=False)

train_dataset_M_0.1.csv and test_dataset_M_0.1.csv saved


In [10]:
def check_nan_impact_on_labels(df, columns_to_check, label_columns):
    # 初始化一个字典来存储结果
    results = []

    for column in columns_to_check:
        # 筛选出指定列为空值的行
        nan_rows = df[df[column].isnull()]

        for label_column in label_columns:
            # 统计标签列中每个值的数量
            value_counts = nan_rows[label_column].value_counts()
            for value, count in value_counts.items():
                results.append({
                    'column_with_nan': column,
                    'label_column': label_column,
                    'label_value': value,
                    'count': count
                })

    return pd.DataFrame(results)

In [20]:
# 统计空值对标签列的影响
freq = "D"
version = "0.3"
# 读取数据
nan_impact = check_nan_impact_on_labels(result, columns_to_check, label_columns)
# 保存结果到 CSV 文件
nan_impact.to_csv(f"{dataset_training_path}nan_impact_{freq}_{version}.csv", index=False)
print(nan_impact)

      column_with_nan                     label_column  label_value  count
0       turnover_rate                    top_or_bottom            0  52484
1       turnover_rate                    top_or_bottom            1  15287
2       turnover_rate                    top_or_bottom            2  15208
3       turnover_rate               top_or_bottom_stat            0  76839
4       turnover_rate               top_or_bottom_stat            1   3178
..                ...                              ...          ...    ...
265  free_share_ratio          top_or_bottom_optimized            2  15433
266  free_share_ratio          top_or_bottom_optimized            1  15387
267  free_share_ratio  top_bottom_volatility_optimized            0  97741
268  free_share_ratio  top_bottom_volatility_optimized            1   2422
269  free_share_ratio  top_bottom_volatility_optimized            2   2061

[270 rows x 4 columns]


In [22]:
freq = "D"
stat_label_count(result, freq, dataset_training_path, version=0.2)

统计结果已保存为 value_counts.csv


In [26]:
version = "0.5"
# 读取数据
nan_impact = check_nan_impact_on_labels(result, columns_to_check, label_columns)
# 保存结果到 CSV 文件
nan_impact.to_csv(f"{dataset_training_path}nan_impact_{freq}_{version}.csv", index=False)
print(nan_impact)

   column_with_nan                     label_column  label_value    count
0               pe                    top_or_bottom            0   919271
1               pe                    top_or_bottom            1   322705
2               pe                    top_or_bottom            2   321889
3               pe               top_or_bottom_stat            0  1436925
4               pe               top_or_bottom_stat            1    66020
5               pe               top_or_bottom_stat            2    60920
6               pe       top_bottom_volatility_stat            0  1480356
7               pe       top_bottom_volatility_stat            1    44401
8               pe       top_bottom_volatility_stat            2    39108
9               pe     top_or_bottom_stat_optimized            0  1454086
10              pe     top_or_bottom_stat_optimized            1    57233
11              pe     top_or_bottom_stat_optimized            2    52546
12              pe          top_or_bot

In [27]:
stat_label_count(result, freq, dataset_training_path, version=0.4)

统计结果已保存为 value_counts.csv


In [28]:
result.to_csv(f"{dataset_training_path}mapped_entry_{freq}_cleaned.csv", index=False)